# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here, we enumerate all record sets and for each record set, we list the field `@id`s, names, and data types.

In [ ]:
import json

# Get the JSON-LD metadata structure
json_metadata = dataset.metadata.to_json()

# List all record sets (@id and name)
record_sets = json_metadata.get('recordSet', [])

if not record_sets:
    print("No record sets found in top-level 'recordSet'. Attempting to extract from 'hasPart'...")
    # Sometimes, record sets are nested under 'hasPart'
    record_sets = []
    for obj in json_metadata.get('hasPart', []):
        if isinstance(obj, dict) and obj.get('@type') == 'RecordSet':
            record_sets.append(obj)
else:
    # If only @id's are listed instead of inlined objects, fetch them from 'included' list
    # First make a lookup table of all included objects
    included = {item['@id']: item for item in json_metadata.get('included', []) if isinstance(item, dict) and '@id' in item}
    temp_rs = []
    for rs in record_sets:
        if isinstance(rs, dict):
            temp_rs.append(rs)
        elif isinstance(rs, str) and rs in included:
            temp_rs.append(included[rs])
    record_sets = temp_rs

if not record_sets:
    print("No record sets could be found with standard keys.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        rs_id = rs.get('@id', 'N/A')
        print(f"- Record Set @id: {rs_id}")
        print(f"  Name: {rs.get('name', rs_id)}")
        if 'field' in rs:
            print("  Fields:")
            fields = rs['field']
            if isinstance(fields, list):
                for field in fields:
                    if isinstance(field, dict):
                        fid = field.get('@id', 'N/A')
                        print(f"    - @id: {fid}, name: {field.get('name','')}, type: {field.get('dataType','')}")
                    else:
                        print(f"    - @id: {field}")
            else:
                print(f"    - {fields}")
        elif 'fields' in rs:
            print("  Fields:")
            for field in rs['fields']:
                if isinstance(field, dict):
                    fid = field.get('@id', 'N/A')
                    print(f"    - @id: {fid}, name: {field.get('name','')}, type: {field.get('dataType','')}")
                else:
                    print(f"    - @id: {field}")
        print()

# Save record set ids and a field id for the next cell, if found
record_set_ids = [rs.get('@id') for rs in record_sets if '@id' in rs]
fields_per_rs = {}
for rs in record_sets:
    fidlist = []
    if 'field' in rs:
        if isinstance(rs['field'], list):
            for f in rs['field']:
                if isinstance(f, dict) and '@id' in f:
                    fidlist.append(f['@id'])
                elif isinstance(f, str):
                    fidlist.append(f)
        elif isinstance(rs['field'], dict):
            fidlist.append(rs['field'].get('@id', ''))
        elif isinstance(rs['field'], str):
            fidlist.append(rs['field'])
    fields_per_rs[rs.get('@id','')] = fidlist

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare to extract data for all record sets found
dataframes = {}
if len(record_set_ids) == 0:
    print("No record sets detected for extraction.")
else:
    for record_set_id in record_set_ids:
        print(f"Extracting records for record set: {record_set_id} ...")
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} rows; columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record set {record_set_id}.")

# Choose one record set as primary for demonstration (first one)
if len(record_set_ids) > 0 and record_set_ids[0] in dataframes:
    display_df_id = record_set_ids[0]
    print(f"\nSample records from record set '{display_df_id}':")
    display_df = dataframes[display_df_id]
    display(display_df.head())
else:
    print("No data loaded for further demo.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps such as filtering records by numeric field values, normalizing numeric columns, and grouping by categorical fields.

Below, we demonstrate filtering and normalization for a numeric field, and grouping by a categorical field.

In [ ]:
# We need at least one DataFrame with numeric and categorical fields

import numpy as np

if len(dataframes) == 0:
    print("No data available for EDA.")
else:
    # Try to automatically select a numeric field and a group field from the first record set
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]

    # Try to guess numeric fields
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_fields:
        # Try to coerce possible numeric columns (sometimes string columns in Croissant)
        for col in df.columns:
            df[col+'_num'] = pd.to_numeric(df[col], errors='coerce')
        numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]
    if not numeric_fields:
        print("Could not find any numeric fields in the data.")
        # Pick the first column as an example (will fail below, but demo code structure)
        numeric_field = df.columns[0]
    else:
        numeric_field = numeric_fields[0]

    print(f"Numeric field selected: {numeric_field}")

    # Filtering step
    threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

    # Group by a likely categorical field (heuristic: string dtype, not numeric)
    group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following cell demonstrates a histogram and a boxplot for a selected numeric field, as well as a violin plot grouped by a categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) == 0:
    print("No data available for visualization.")
else:
    df_plot = df.copy()
    # Numeric field already detected in previous cell
    plt.figure(figsize=(14,4))
    plt.subplot(1,2,1)
    sns.histplot(df_plot[numeric_field].dropna(), bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)

    plt.subplot(1,2,2)
    sns.boxplot(y=df_plot[numeric_field])
    plt.title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()

    # Grouped violin plot
    if group_fields:
        group_field = group_fields[0]
        plt.figure(figsize=(10,6))
        sns.violinplot(x=group_field, y=numeric_field, data=df_plot)
        plt.title(f'{numeric_field} distribution by {group_field}')
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset describing mass spectrometry analysis of extracellular vesicles from human mast cells. We loaded the structured data using `mlcroissant`, examined the available record sets and fields, extracted tabular data, performed basic exploratory statistical analysis, and visualized the distributions and groupings present in the records. For further analyses, we suggest investigating specific proteins of interest, abundance trends, or linking with metadata from UniProt annotations.